In [14]:
from datasets import load_dataset

# format: instruction, input, output
noralpaca = load_dataset("NbAiLab/norwegian-alpaca")["train"]
nor_fleurs = load_dataset("RuterNorway/Fleurs-Alpaca-EN-NO")["train"]
nor_orca = load_dataset("RuterNorway/OpenOrcaNo-15k")["train"]

In [36]:
from datasets import concatenate_datasets, DatasetDict

dataset = concatenate_datasets([noralpaca, nor_fleurs, nor_orca])
dataset = dataset.remove_columns(["instruction_en", "input_en", "output_en"])
dataset.shuffle()

dataset = dataset.train_test_split(test_size=0.01, shuffle=True)

In [37]:
import re

train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

# fjern "du er en ai-assistent" 
def remove_starting_du_er(text):
    return re.sub(r"^Du er .*?\.", "", text).strip()

# update the "instruction" column
train_df["instruction"] = train_df["instruction"].apply(remove_starting_du_er)
test_df["instruction"] = test_df["instruction"].apply(remove_starting_du_er)


In [38]:
train_df = train_df[~train_df["output"].isna()]
test_df = test_df[~test_df["output"].isna()]


In [42]:
from datasets import Dataset
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df).remove_columns("__index_level_0__"),
    "test": Dataset.from_pandas(test_df)
})
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 67714
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 684
    })
})

In [43]:
dataset.push_to_hub("tollefj/nor-instruct")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/68 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/473 [00:00<?, ?B/s]